# Suite LIF — Lifetimes: every read within a transaction

Every read of a store's data is made within a transaction, as every write is ([CAP-315: every read within a transaction or through a follow](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-315.md#decided)). A visitor of the store's
data, an object and each visitor read from it (its properties, its adjacencies, its entries, the value objects and
containers it holds), lives only within the transaction that made it, the outermost open on its store; used after it
ends, it is refused. The store lives on, and so do a path, a reference and every value a program copies out within a
transaction, which is stale from the moment the transaction ends. Within a transaction, the store gives one visitor of
each object. The durable object the store holds is wrapped by each transaction's visitor of it, which reads and
writes it through the transaction's proposals. A frame `store.transaction` gives back holds bindings and assignments,
not visitors: each symbol's object by its reference, and the values the transaction left in the properties its specs
name; it refuses a counted object, which has no reference. Schema objects are not covered.

Python and TypeScript run this suite with the same case IDs. The case setup is below.

In [ ]:
import { Builders as B, Plain, Proxies, References } from "@mbse/schemas/Framework";
import { LookupError, ValueError } from "@mbse/schemas/Framework/Errors";
import { assert, equal, map, raises } from "./support.js";

/** crm.Person, owned by the singleton crm.Book by key, with a home address (a value object) and tags (a container);
 * crm.Note, counted, about a person; crm.Tag, counted, linking nothing. */
function crm(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("lif/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Address: S.OfObject, Person: S.OfObject, Holds: S.OfRelation, Book: S.OfObject, Note: S.OfObject,
      About: S.OfRelation, Tag: S.OfObject })
    .spec([
      (_: any) => _.Address.name("crm.Address").properties({ city: S.OfStr }),
      (_: any) => _.Person.name("crm.Person").ref(true).properties({
        name: S.OfStr, home: _.Address,
        tags: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr])) }).adjacencies([
        (t: any) => t.name("book").relation([_.Holds]).me("contact").owner(true)]),
      (_: any) => _.Holds.name("crm.Holds").links([(t: any) => t.name("book"), (t: any) => t.name("contact")]).properties({
        key: S.OfStr }).uniques([["book", "key"], ["contact"]]),
      (_: any) => _.Book.name("crm.Book").ref(true).singleton("crm.TheBook").adjacencies({
        contacts: (t: any) => t.relation([_.Holds]).me("book") }),
      (_: any) => _.Note.name("crm.Note").ref(true).properties({ text: S.OfStr }).adjacencies({
        about: (t: any) => t.relation([_.About]).me("note") }),
      (_: any) => _.About.name("crm.About").links([(t: any) => t.name("note"), (t: any) => t.name("person")]),
      (_: any) => _.Tag.name("crm.Tag").ref(true).properties({ label: S.OfStr }),
    ])]))]);
}

const store: any = new B.Store();
store.transaction(crm);
const [Person, Book, Note, Tag] = ["crm.Person", "crm.Book", "crm.Note", "crm.Tag"].map((name) => store.registered(name));
store.transaction((t: any) => t.uid("people").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
  .symbols({ book: Book, ada: Person, bob: Person }).spec([
    (_: any) => _.book.contacts({ ada: (e: any) => e.contact(_.ada), bob: (e: any) => e.contact(_.bob) }),
    (_: any) => _.ada.name("Ada"),
    (_: any) => _.bob.name("Bob"),
  ])]))]));
const ADA = map({ singleton: "crm.TheBook", next: { entry: { i: "contacts", o: "book", properties: { key: "ada" },
  next: { target: {} } } } });

/** Asserts that `use`, a use of a visitor whose transaction has ended, is refused. */
function refused(use: () => unknown): void {
  raises(ValueError, use, "a visitor lives only within the transaction that made it");
}

<a id="LIF-01"></a>
## LIF-01 · A visitor lives within the transaction that made it

An object, its property's visitor, its adjacency and an entry of it, read within a transaction, are refused once it
ends, each whatever the use: a member read, a conversion, iteration, its length, its representation. What the program
copied out within it is its own.

In [ ]:
let book: any, contacts: any, entry: any, ada: any, name: any, harvested = "";
store.transaction(() => {
  book = store.singleton("crm.TheBook");
  contacts = book.contacts;
  entry = contacts.at("ada");
  ada = entry.contact;
  name = ada.name;
  harvested = `${name}`;
  assert(harvested === "Ada" && contacts.at("ada").contact === ada && contacts[0] === entry); // one visitor of each
});
for (const use of [() => book.contacts, () => ada.name, () => `${name}`, () => name.get(), () => entry.contact,
  () => entry.key, () => [...contacts], () => contacts.length, () => contacts[0], () => String(ada),
  () => ada.as_schema()]) {
  refused(use);
}
assert(harvested === "Ada"); // stale from the moment the transaction ended, but the program's

<a id="LIF-02"></a>
## LIF-02 · A store is read only within a transaction

`store.singleton(...)` and `store.extent(...)` are refused with no transaction open. Each transaction gives visitors of
its own: the singleton again, another visitor of the same object, the earlier one refused. A visitor of a transaction
given to a later one, to update it or to link it, is refused.

In [ ]:
for (const use of [() => store.singleton("crm.TheBook"), () => store.extent("crm.Person")]) {
  raises(ValueError, use, "a store is read only within a transaction (store.transaction)");
}
store.transaction(() => {
  const again = store.singleton("crm.TheBook");
  assert(again !== book && again === store.singleton("crm.TheBook")); // one visitor of it, in this transaction
  assert(equal(store.extent("crm.Person").map((person: any) => `${person.name}`).sort(), ["Ada", "Bob"]));
  raises(ValueError, () => store.crm.Person(ada, (p: any) => p.name("Ada Lovelace")), // a visitor of the earlier one
    "a visitor lives only within the transaction that made it");
  raises(ValueError, () => store.crm.Note((n: any) => n.about([ada])),
    "a visitor lives only within the transaction that made it");
});

<a id="LIF-03"></a>
## LIF-03 · A value object and its visitor

A value object held by nothing, built standalone, is data: read anywhere, and given to an object, copied, so that it
stays the program's. Read through the object that holds it, a value object, or a container, is a visitor of the
transaction, owned by that object, and refused once the transaction ends.

In [ ]:
const paris = store.builder("crm.Address").city("Paris").create(); // no transaction: a standalone value, data
assert(`${paris.city}` === "Paris" && Proxies.owner_of(paris) === null);
let home: any, held: any;
store.transaction(() => {
  const ada = store.singleton("crm.TheBook").contacts.at("ada").contact;
  store.crm.Person(ada, (p: any) => p.home(paris).tags(["math", "poetry"]));
  ada.home.as_object((value: any) => (home = value));
  ada.tags.as_value((a: any) => a.as_container((value: any) => (held = value)));
  assert(`${home.city}` === "Paris" && home !== paris && Proxies.owner_of(home) === ada); // the object's copy
  assert(held.length === 2 && Proxies.owner_of(paris) === null); // the value given stays the program's
});
for (const use of [() => home.city, () => held.length, () => Proxies.owner_of(home)]) refused(use);
assert(`${paris.city}` === "Paris"); // data, read with no transaction open

<a id="LIF-04"></a>
## LIF-04 · Frames hold bindings and assignments, not visitors

A frame `store.transaction` gives back binds each symbol to its object's reference, a value read anywhere, and
assigns, by symbol, the values the transaction left in the properties its specs name. A symbol bound to a counted
object, which has no reference, is refused, and assigns nothing. `References.of` gives a frame's reference, its
assignments among it.

In [ ]:
const [frame] = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ who: Person }).spec([(_: any) => _.who.name("Ada")])]))]));
const reference = frame.get("who");
assert(equal(Plain.ToPlain(store)(reference.as_schema(), reference), map({ path: ADA }))); // read with no transaction open
assert(equal(frame.assigns, map({ who: { name: "Ada" } })) && frame.errors.length === 0);
const [noted] = store.transaction((t: any) => t.mutations([(mu: any) => mu.map((m: any) => m.symbols([(p: any) => p
  .symbols({ who: Person }).spec([(_: any) => _.who.name("Ada")])]).then([(n: any) => n.upsert((u: any) => u.key("n/1")
  .exists([(p: any) => p.symbols({ who: Person, note: Note }).spec([(_: any) => _.note.text("hi").about([_.who])])]))]))]));
assert(equal([...noted.keys()], ["who", "note"]) && equal(noted.assigns, map({ who: { name: "Ada" } })));
raises(LookupError, () => noted.get("note"), "symbol 'note' of the frame: a counted 'crm.Note' has no reference");
const refusedNote = [...noted.values()][1]; // what the frame holds of it: refused whatever the use
assert(String(refusedNote) === "<refused: a counted object>");
const written = References.of(store, frame);
assert(equal(Plain.ToPlain(store)(written.as_schema(), written), map({ binds: { who: { path: ADA } }, assigns: [
  { object: { path: ADA }, properties: { name: { str: "Ada" } } }], scope: "0" }))); // its input frame's id
raises(LookupError, () => References.of(store, noted), "symbol 'note' of the frame: a counted 'crm.Note' has no reference");

<a id="LIF-05"></a>
## LIF-05 · What a decoder gives back

A document read within a transaction gives visitors of it; read with none open, the decoder's own transaction makes
them, and what it gives back is refused.

In [ ]:
const document = store.transaction(() => Plain.ToPlain(store).Reachable(Tag, store.crm.Tag((t: any) => t.label("urgent"))));
const read: any = Plain.FromPlain(store).Reachable(Tag, document); // its own transaction, ended
refused(() => read.label);
store.transaction(() => {
  assert(`${(Plain.FromPlain(store).Reachable(Tag, document) as any).label}` === "urgent");
});

<a id="LIF-06"></a>
## LIF-06 · Expiry follows the transaction that made a visitor

A transaction's expression built with no transaction open is no transaction's, and lives on; one built within a
transaction is that transaction's, and is refused once it ends, whatever its class.

In [ ]:
const expression = store.Mutations.OfTransaction((t: any) => t.uid("later")); // built with none open
let within: any;
store.transaction(() => {
  within = store.Mutations.OfTransaction((t: any) => t.uid("within"));
  assert(`${within.uid}` === "within");
});
assert(`${expression.uid}` === "later");
refused(() => within.uid);
store.transaction(expression); // executed later